# 冻结基础模型的现金结构
0拟合、0回测。重建冻结网络并与来源未平滑预测逐行核对，分离softmax现金、15%现金底线和SOXL上限溢出。零标准化输入仅诊断，不选参数、不构造交易策略。

In [1]:
import os,json,qlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/allocator_cash_geometry_20261014'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
from src.workflow.allocator_cash_geometry import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
summary

{'experiment': 'allocator_cash_geometry_20261014',
 'plan_sha256': '4f605aa02d73787eb9bc6a519ce5d39f0e9f14ae13d9254e7c66c34c01c809f4',
 'decision': 'diagnostic_only_no_promotion',
 'objective_achieved': False,
 'new_fits': 0,
 'new_backtests': 0,
 'sessions': 944,
 'first_signal': '2023-01-03 00:00:00',
 'last_signal': '2026-10-07 00:00:00',
 'prediction_max_absolute_error': 5.7676443732113114e-08,
 'standardized_zero_input_weights': [0.30000001192092896,
  0.4749508202075958,
  0.22504912316799164],
 'zero_logits_legal_weights': [0.28333333134651184,
  0.28333333134651184,
  0.43333330750465393],
 'recorder_id': '2b6c68ba4a00474d933e0b6170fd5434',
 'experiment_id': '747604294599644737',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/allocator_cash_geometry_20261014/qlib_recorder',
 'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/allocator_cash_geometry_20261014/qlib_recorder/747604294599644737/2b6c68ba4a00474d933e0b6170fd5434',
 'provider_uri': '/Users/rziiiii/de

# 基础模型现金结构与目标核查

0拟合、0回测。读取冻结f696来源网络状态、原始特征及训练期scaler，重建未平滑推理；944个信号日期为2023-01-03至2026-10-07，覆盖当前参考账户共同窗口。三资产权重、tail及alpha输出与来源Recorder逐行核对，最大绝对误差5.77e-8，处于float32数值精度范围。

## 现金不是单一来源

|信号年|现金底线%|softmax分配的现金百分点|SOXL上限转入现金百分点|最终模型现金%|上限触及天数|
|---|---:|---:|---:|---:|---:|
|all|15.00|23.44|2.35|40.79|419/944|
|2023|15.00|20.63|1.58|37.20|105/250|
|2024|15.00|34.28|0.93|50.22|62/252|
|2025|15.00|23.94|2.07|41.01|106/250|
|2026|15.00|12.22|5.56|32.79|146/192|

公式：softmax在三个资产间分配85%权重，再给现金加15%底线；若SOXL超过30%，超出部分按当前SOXX与现金比例重新分配，最后归一化。平均40.79%现金中，15个百分点是底线，23.44个百分点来自网络softmax，2.35个百分点来自内部SOXL上限。2026年的上限转入现金较多（5.56个百分点），但它不能解释全期主要现金部分。

419/944日触及的是网络内部上限；前一卡“Stage1合法投影不再增加现金”指后续另一层，两者不矛盾。均值来源按signal_date年度划分，与前一卡按trade_date年度划分在年界相差一日，不能将年度数值直接视作修订。

零logit合法权重为SOXL/SOXX/CASH约28.33/28.33/43.33%；但冻结模型的标准化零输入输出为30.00/47.50/22.50%。因此实际40.79%现金不能简单归因为“模型没学到东西，仍保持均匀初始化”。零输入是人工参考点，不是最优仓位、真实常态或可交易对照，也不能将18个百分点差额视为可回收收益。

## 已有训练目标

f696选择记录明确为`pit_only_aux_005 / relative_return_baseline`。代码的基础配置损失是：负的年度平均相对SOXX对数收益，加0.5倍年度均值标准差，再加0.006倍平均换手。组合收益中另扣10bp训练成本；共享编码器另有主标的尾部/上涨分类与辅助样本分类损失。固定候选的额外绝对回撤、条件回撤、5日短缺及直接现金目标惩罚均为0。

所以“原模型没有使用SOXX基准”不成立。0.006是每单位换手的偏好惩罚，数值相当于60bp的损失系数，**不是IBKR佣金、不是实际成交费，也不能在回测收益里再次扣除**。它和已计入的10bp训练成本作用不同。当前账户按Tiered结算并不代表旧模型已按Tiered重新训练；本卡没有因此假定重训会改善超额。

## 决定与下一步

决定：diagnostic_only_no_promotion。内部投影确实增加现金，但只有约2.35个百分点；不据此放宽SOXL上限或把现金强制转入SOXX。主体现金决策来自网络与其既有目标/输入的共同作用。本卡没有测量各训练损失的因果影响，不将系数大小直接当成主导程度。

下一项机制应针对同结构基础分配器中“额外换手惩罚”的经济作用，先核查是否已有同输入、同初始化的固定对照；若已有负面证据则不重复。不得把旧动作层费用一致性实验混作基础网络训练对照。任何候选仍需独立费用账户、30%回撤、年度/滚动/区块超额门和独立前向资格。

来源经Qlib Recorder校验并保存到NAS；[现金分解](cash_geometry.csv)、[summary](summary.json)和[对象清单](verification_manifest.json)为入口。复用历史与价格/成交资格限制继承父证据。稳定净SOXX超额目标未完成，生产SIM未修改。
